In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
llm = ChatOpenAI(model="gpt-4o-mini")

llm.invoke([HumanMessage("잘 지냈어?")])

AIMessage(content='네, 잘 지냈습니다! 당신은 어떻게 지내고 계신가요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 12, 'total_tokens': 30, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_29330a9688', 'id': 'chatcmpl-CxTLwrhcIc1uOwgpmAxEOWMGUdyH9', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019bb64f-7747-7b50-abe6-4de420b8b870-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 18, 'total_tokens': 30, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [2]:
from langchain_core.tools import tool
from datetime import datetime
import pytz



@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각을 반환하는 함수
    
    
    Args:
        timezone (str): 타임존(예: 'Asia/Seoul'). 실제 존재해야함
        location (str): 지역명. 타임존은 모든 지명에 대응되지 않으므로 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재 시각 {now}'
    print(location_and_local_time)
    return location_and_local_time

In [3]:
tools = [get_current_time,]
tools_dict = {"get_current_time": get_current_time, }

llm_with_tools = llm.bind_tools(tools)

In [4]:
from langchain_core.messages import SystemMessage


messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇 시야?")
]

response = llm_with_tools.invoke(messages)
messages.append(response)

print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇 시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 130, 'total_tokens': 153, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c4585b5b9c', 'id': 'chatcmpl-CxTLyq1fWhEdkbYh5o3Q3upiTxmKg', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bb64f-7f51-7861-8808-564f74f11992-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_hKBsEp0bT7NitV1uBvI7NQW2', 'type': 'tool_call'}]

In [5]:
for tool_call in response.tool_calls:
    selected_tool = tools_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재 시각 2026-01-13 16:43:53


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇 시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 130, 'total_tokens': 153, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c4585b5b9c', 'id': 'chatcmpl-CxTLyq1fWhEdkbYh5o3Q3upiTxmKg', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bb64f-7f51-7861-8808-564f74f11992-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_hKBsEp0bT7NitV1uBvI7NQW2', 'type': 'tool_call'

In [6]:
llm_with_tools.invoke(messages)

AIMessage(content='부산은 현재 2026년 1월 13일 16시 43분입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 186, 'total_tokens': 209, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c4585b5b9c', 'id': 'chatcmpl-CxTMAuKfmbuHSpvde7vVyCM8iaAtg', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019bb64f-aee6-7ea3-9235-ceb5e96facbf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 186, 'output_tokens': 23, 'total_tokens': 209, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [7]:
from pydantic import BaseModel, Field


class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: AAPL)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")

In [8]:
import yfinance as yf

@tool
def get_yf_stock_history(ticker:str, period: str="1mo") -> str:
    """ 주식 종목의 가격 데이터를 조회하는 함수
    
    Args:
        ticker (str): 주식 티커 심볼 (예: 'TSLA', 'AAPL', 'MSFT').
        period (str): 조죄할 기간 (예: '1d', '5d', '1mo', '3mo', '1y').
    """
    try:
        stock = yf.Ticker(ticker)
        history = stock.history(period=period)

        if history.empty:
            return f"Error: '{ticker}'에 대한 데이터를 찾을 수 없습니다."
        
        return history.to_markdown()
    
    except Exception as e:
        return f"Error: {str(e)}"



tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [9]:
messages.append(HumanMessage("테슬라는 한 달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
print(response)
messages.append(response)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 309, 'total_tokens': 332, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c4585b5b9c', 'id': 'chatcmpl-CxTMIcCUlLzLTyDWbiUNhBICsfLKq', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019bb64f-cfd0-7211-830d-940911e9c5f8-0' tool_calls=[{'name': 'get_yf_stock_history', 'args': {'ticker': 'TSLA', 'period': '1mo'}, 'id': 'call_Z6E7FjoxAwCm15V3o06CaZoJ', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 309, 'output_tokens': 23, 'total_tokens': 332, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [10]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    print(tool_msg)
    

{'ticker': 'TSLA', 'period': '1mo'}
content="Error: 'NoneType' object is not subscriptable" name='get_yf_stock_history' tool_call_id='call_Z6E7FjoxAwCm15V3o06CaZoJ'


In [11]:
llm_with_tools.invoke(messages)

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 355, 'total_tokens': 378, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c4585b5b9c', 'id': 'chatcmpl-CxTMKE24yOYLd718pRtWu3V8V5Mx4', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bb64f-d5ec-7283-9f79-e197c4b1258a-0', tool_calls=[{'name': 'get_yf_stock_history', 'args': {'ticker': 'TSLA', 'period': '1mo'}, 'id': 'call_MJztktmpWPz5OAYKscOB0d1U', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 355, 'output_tokens': 23, 'total_tokens': 378, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'r

In [12]:
for c in llm.stream([HumanMessage("잘 지냈어? 한국 사회의 문제점이 무엇인지 이야기해줘.")]):
    print(c.content, end='|')

|안|녕하세요|!| 한국| 사회|의| 문제|점|에| 대해| 여러| 가지| 측|면|에서| 이야기|할| 수| 있습니다|.| 다음|은| 현재| 한국| 사회|에서| 중요한| 문제|들| 중| 일부|입니다|.

|1|.| **|저|출|산| 및| 고|령|화|**|:| 한국|은| 세계|에서| 가장| 낮|은| 출|산|율|을| 기록|하고| 있으며|,| 이|로| 인해| 고|령|화| 사회|가| 빠|르게| 진행|되고| 있습니다|.| 이는| 경제| 성장|과| 사회| 복|지| 시스템|에| 큰| 부담|을| 주|고| 있습니다|.

|2|.| **|청|년| 실|업|**|:| 청|년|층|의| 취|업|난|은| 여|전히| 심|각|한| 문제|로| 남|아| 있습니다|.| 졸|업| 후| 취|업|에| 어려|움을| 겪|는| 사례|가| 많|고|,| 이|로| 인해| 젊|은| 세|대|의| 삶|의| 질|이| 저|하|되고| 있습니다|.

|3|.| **|부|동|산| 문제|**|:| 주|택| 가격| 상승|과| 전|세|난|은| 많은| 사람|들에게| 큰| 부담|을| 주|고| 있습니다|.| 집|을| 구|하기| 어려|운| 젊|은| 세|대|와| 무|주|택| 서|민|들의| 고|통|이| 심|화|되고| 있습니다|.

|4|.| **|사회|적| 불|평|등|**|:| 소|득| 격|차|와| 사회|적| 불|평|등|이| 심|각|한| 문제|입니다|.| 특히|,| 대|기업|과| 중|소|기업| 간|의| 임|금| 차|이|,| 지역| 간|의| 발전| 차|이| 등이| 문제가| 되고| 있습니다|.

|5|.| **|정|신| 건강| 문제|**|:| 정신| 건강| 문제|에| 대한| 인|식|이| 개선|되고| 있지만|,| 여|전히| 많은| 사람들이| 도움|을| 받|기| 어려|운| 상황|입니다|.| 스트|레스|,| 우|울|증|,| 불|안|장|애| 등이| 증가|하고| 있습니다|.

|6|.| **|성| 차|별|과| 성|폭|력|**|:| 성|별|에| 따른| 차|별|과| violence|가| 여|전히| 존재|합니다|.| 성|폭|력| 사건|과| 관련|한| 사회|적| 

In [13]:
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇 시야?")
]

response = llm_with_tools.stream(messages)

is_first = True

for chunk in response:
    print("chunk type: ", type(chunk))
    if is_first:
        is_first = False
        gathered = chunk
    else:
        gathered += chunk

    print("content: ", gathered.content, "tool_call_chunk", gathered.tool_calls)

messages.append(gathered)

chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_tx4L61B9zN2MeyHMSzbXPqPL', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_tx4L61B9zN2MeyHMSzbXPqPL', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_tx4L61B9zN2MeyHMSzbXPqPL', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {'timezone': ''}, 'id': 'call_tx4L61B9zN2MeyHMSzbXPqPL', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {'timezone': 'Asia'}, 'id': 'call_tx4L61B9zN2MeyHMSzbXPqPL', 'type': 'tool_c

In [14]:
gathered

AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai', 'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c4585b5b9c', 'service_tier': 'default'}, id='lc_run--019bb656-266b-7670-99f3-3a07bf6e6840', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_tx4L61B9zN2MeyHMSzbXPqPL', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 230, 'output_tokens': 23, 'total_tokens': 253, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, tool_call_chunks=[{'name': 'get_current_time', 'args': '{"timezone":"Asia/Seoul","location":"부산"}', 'id': 'call_tx4L61B9zN2MeyHMSzbXPqPL', 'index': 0, 'type': 'tool_call_chunk'}], chunk_position='last')

In [15]:
for tool_call in gathered.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]

    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재 시각 2026-01-13 16:54:23


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇 시야?', additional_kwargs={}, response_metadata={}),
 AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai', 'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c4585b5b9c', 'service_tier': 'default'}, id='lc_run--019bb656-266b-7670-99f3-3a07bf6e6840', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_tx4L61B9zN2MeyHMSzbXPqPL', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 230, 'output_tokens': 23, 'total_tokens': 253, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, tool_call_chunks=[{'name': 'get_current_time', 'args': '{"timezone":"Asia/Seoul","location":"부산"}', 'id': 'call_tx4L61B9zN2MeyHMSzbXPqPL', 'index': 0, 't

In [16]:
for c in llm_with_tools.stream(messages):
    print(c.content, end='|')

|부|산|의| 현재| 시|각|은| |202|6|년| |1|월| |13|일| |16|시| |54|분| |23|초|입니다|.||||